In [0]:
#Kingdom Data & Statistics

import pandas as pd
from pyspark.sql.functions import col, count, when, isnan

#Creating the Tableau Directoey 
dbutils.fs.mkdirs("/Volumes/workspace/default/museum/tableau/")

# Load data
df = spark.read.parquet("/Volumes/workspace/default/museum/parquet/")

# Kingdom distribution
kingdom_dist = df.groupBy("kingdom").count().orderBy("count", ascending=False)
kingdom_dist.toPandas().to_csv("/Volumes/workspace/default/museum/tableau/kingdom_distribution.csv", index=False)

# Collection code distribution
collection_dist = df.groupBy("collectionCode", "kingdom").count().orderBy("count", ascending=False)
collection_dist.toPandas().to_csv("/Volumes/workspace/default/museum/tableau/collection_distribution.csv", index=False)

# Continent distribution
continent_dist = df.groupBy("continent", "kingdom").count().orderBy("count", ascending=False)
continent_dist.toPandas().to_csv("/Volumes/workspace/default/museum/tableau/continent_distribution.csv", index=False)


In [0]:
#Data Quality Statistics 

# Null counts for all key columns
key_cols = ["kingdom", "continent", "country", "basisOfRecord", 
            "collectionCode", "year", "order", "family", "phylum"]

from pyspark.sql.functions import sum as spark_sum

null_counts = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in key_cols
])

null_pd = null_counts.toPandas().T.reset_index()
null_pd.columns = ["column", "null_count"]
null_pd["total_rows"] = df.count()
null_pd["null_percentage"] = (null_pd["null_count"] / null_pd["total_rows"] * 100).round(2)
null_pd.to_csv("/Volumes/workspace/default/museum/tableau/data_quality.csv", index=False)

# Records over time
year_dist = df.filter(col("year") != "Unknown").groupBy("year", "kingdom").count().orderBy("year")
year_dist.toPandas().to_csv("/Volumes/workspace/default/museum/tableau/records_over_time.csv", index=False)

print("Dashboard 1 & 2 quality data exported successfully")

Dashboard 1 & 2 quality data exported successfully


In [0]:
#Model Performance Results

# Model performance summary
model_results = pd.DataFrame({
    "Model": ["Random Forest", "Decision Tree", "Logistic Regression", "GBT Binary", "Tuned Random Forest"],
    "Accuracy": [0.9998, 0.9987, 0.9601, 0.9996, 0.9998],
    "F1_Score": [0.9998, 0.9987, 0.9532, 0.9996, 0.9998],
    "Precision": [0.9998, 0.9987, 0.9612, 0.9996, 0.9998],
    "Recall": [0.9998, 0.9987, 0.9601, 0.9996, 0.9998],
    "Framework": ["MLlib", "MLlib", "MLlib", "MLlib", "MLlib"]
})
model_results.to_csv("/Volumes/workspace/default/museum/tableau/model_performance.csv", index=False)

# Feature importance
feature_importance = pd.DataFrame({
    "Feature": ["subDepartment", "phylum", "order", "family", "basisOfRecord",
                "collectionCode", "continent", "country", "year", 
                "individualCount", "sex", "typeStatus", "month", "lifeStage"],
    "Importance": [0.370981, 0.210141, 0.193041, 0.085091, 0.041654,
                   0.033732, 0.022865, 0.019333, 0.008757,
                   0.005588, 0.003617, 0.003436, 0.001493, 0.000272]
})
feature_importance.to_csv("/Volumes/workspace/default/museum/tableau/feature_importance.csv", index=False)

# Sklearn vs MLlib comparison
sklearn_comparison = pd.DataFrame({
    "Model": ["Random Forest", "Decision Tree", "Logistic Regression"],
    "MLlib_Accuracy": [0.9998, 0.9987, 0.9601],
    "Sklearn_Accuracy": [0.9670, 0.9784, 0.6715]
})
sklearn_comparison.to_csv("/Volumes/workspace/default/museum/tableau/sklearn_vs_mllib.csv", index=False)

print("Dashboard 2 model performance data exported successfully")

Dashboard 2 model performance data exported successfully


In [0]:
#Business Insights & Scalability 

# Geographic spread - top countries per kingdom
geo_data = df.filter(col("country") != "Unknown") \
    .groupBy("country", "kingdom", "continent") \
    .count() \
    .orderBy("count", ascending=False) \
    .limit(100)
geo_data.toPandas().to_csv("/Volumes/workspace/default/museum/tableau/geographic_spread.csv", index=False)

# Specimen types
specimen_types = df.groupBy("basisOfRecord", "kingdom").count().orderBy("count", ascending=False)
specimen_types.toPandas().to_csv("/Volumes/workspace/default/museum/tableau/specimen_types.csv", index=False)

# Scalability data
scalability = pd.DataFrame({
    "Data_Fraction": [0.25, 0.50, 0.75, 1.0],
    "Row_Count": [285000, 570000, 855000, 1143438],
    "Processing_Time_Seconds": [15, 28, 42, 57]
})
scalability.to_csv("/Volumes/workspace/default/museum/tableau/scalability.csv", index=False)

print("Dashboard 3 & 4 data exported successfully")

Dashboard 3 & 4 data exported successfully


In [0]:
#Verify all Exports

import os

files = dbutils.fs.ls("/Volumes/workspace/default/museum/tableau/")
print("Exported files:")
for f in files:
    print(f"  {f.name} — {f.size:,} bytes")

Exported files:
  collection_distribution.csv — 472 bytes
  continent_distribution.csv — 1,453 bytes
  data_quality.csv — 307 bytes
  feature_importance.csv — 278 bytes
  geographic_spread.csv — 2,627 bytes
  kingdom_distribution.csv — 201 bytes
  model_performance.csv — 300 bytes
  records_over_time.csv — 53,048 bytes
  scalability.csv — 107 bytes
  sklearn_vs_mllib.csv — 127 bytes
  specimen_types.csv — 649 bytes
